# Read Your Way

**A reading tool for neurodivergent learners — HCDE 530, Mini Project 2 (Spring 2026)**

Read Your Way is a small reading companion for neurodivergent students (dyslexia, ADHD, autism) aged roughly 10–16. Mainstream EdTech platforms in India — Byju's, Duolingo, government e-learning apps — offer no cognitive adaptation. They hand every learner the same visually overloaded, comparison-driven interface and expect them to cope. Many simply shut down. Many go without digital learning support entirely.

This notebook is a calmer alternative. For MP2, scope is intentionally narrow: it does two things, and it tries to do them well.

1. **OpenDyslexic font toggle** — switch the reading typeface to OpenDyslexic, a font designed to reduce letter-flipping and visual crowding for dyslexic readers.
2. **Reading pace control** — a guided pacer walks the learner through the passage one chunk at a time, at a speed they choose. The learner controls the page instead of the page controlling them.

The original declaration named six features. The remaining four (contrast control, motion reduction, audio-synced reading, progress milestones) are kept as a roadmap at the bottom of the notebook.

## How to use

1. Run the cells below in order (Shift+Enter).
2. Pick a passage from the dropdown.
3. Toggle **Font** between *Default* and *OpenDyslexic* to feel the difference.
4. Pick a **Mode**:
   - *Read freely* — full passage, your eyes, your pace.
   - *Guided pacing* — one chunk highlights at a time. Press ▶ to auto-advance, or step with the Previous / Next buttons.
5. Use the **Words per minute** slider to set the pacer's speed. Lower is calmer.

Everything is local. Nothing is logged. No streaks. No comparison.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Inject the OpenDyslexic font globally so the renderer can apply it on demand.
# Source: open-dyslexic npm package on jsDelivr (public CDN, free font license).
display(HTML("""
<style>
@font-face {
  font-family: "OpenDyslexic";
  src: url("https://cdn.jsdelivr.net/npm/open-dyslexic@1.0.3/woff/OpenDyslexic-Regular.woff") format("woff"),
       url("https://cdn.jsdelivr.net/gh/antijingoist/open-dyslexic/compiled/OpenDyslexic-Regular.otf") format("opentype");
  font-weight: normal;
  font-style: normal;
  font-display: swap;
}
</style>
"""))
print("Setup complete. OpenDyslexic font registered.")

In [ ]:
# Public-domain Aesop's fables (Townsend / Jacobs translations, lightly adapted).
# Short, age-appropriate, recognizable — good test material for the pacer.
PASSAGES = {
    "The Lion and the Mouse": (
        "A Lion was asleep in his lair when a little Mouse, not seeing him, "
        "ran over his face and woke him. Angry, the Lion caught the Mouse in "
        "his paw and was about to kill her. The Mouse begged for her life, "
        "saying, 'If you spare me, I will one day repay you.' The Lion laughed "
        "at the thought that so small a creature could ever help him, but he "
        "let her go. Not long after, the Lion was caught in a hunter's net. "
        "The Mouse, hearing him roar, ran to the spot and gnawed through the "
        "ropes with her sharp teeth, setting him free. The Lion thanked her, "
        "and learned that even the smallest friend may be the greatest help."
    ),
    "The Tortoise and the Hare": (
        "A Hare was making fun of a Tortoise for being so slow. 'Do you ever "
        "get anywhere?' he asked with a laugh. 'Yes,' said the Tortoise, 'and "
        "faster than you think. Let us race and see.' The Hare thought this so "
        "absurd that he agreed at once. The Fox marked the course, and they "
        "started. The Hare ran far ahead, then stopped to rest, sure he could "
        "overtake the Tortoise whenever he wished. He lay down by the road "
        "and fell asleep. The Tortoise plodded on, never stopping. When the "
        "Hare woke, he saw the Tortoise close to the finish line. He ran his "
        "fastest, but he could not catch up. Slow and steady wins the race."
    ),
    "The Boy Who Cried Wolf": (
        "A Shepherd Boy who watched his flock near a village grew bored, and "
        "to amuse himself he cried out, 'Wolf! Wolf!' though no wolf was "
        "there. The villagers ran to help, only to find him laughing. He "
        "played the same trick again, and again the villagers came running, "
        "and again he laughed. One day a real Wolf came out of the forest "
        "and began to scatter the sheep. The Boy cried in terror, 'Wolf! "
        "Wolf!' But the villagers, thinking it was another trick, paid no "
        "attention. The Wolf ate his fill of the sheep. And the Boy learned "
        "that no one believes a liar, even when he tells the truth."
    ),
}

def split_into_chunks(text, words_per_chunk):
    """Break a passage into pacer-sized chunks of N words each."""
    words = text.split()
    return [
        " ".join(words[i:i + words_per_chunk])
        for i in range(0, len(words), words_per_chunk)
    ]

print(f"Loaded {len(PASSAGES)} passages.")

In [ ]:
# ---------------------------------------------------------------------------
# Controls
# ---------------------------------------------------------------------------
passage_dd = widgets.Dropdown(
    options=list(PASSAGES.keys()),
    description="Passage:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
)

font_toggle = widgets.ToggleButtons(
    options=[("Default sans-serif", "default"), ("OpenDyslexic", "OpenDyslexic")],
    value="default",
    description="Font:",
    style={"description_width": "120px"},
)

mode_toggle = widgets.ToggleButtons(
    options=[("Read freely", "free"), ("Guided pacing", "guided")],
    value="free",
    description="Mode:",
    style={"description_width": "120px"},
)

wpm_slider = widgets.IntSlider(
    value=180, min=60, max=400, step=10,
    description="Words / min:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
    continuous_update=False,
)

chunk_size_slider = widgets.IntSlider(
    value=6, min=2, max=15, step=1,
    description="Words / chunk:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
    continuous_update=False,
)

# Auto-advance pacer (Play widget) bound to a position slider
play_button = widgets.Play(value=0, min=0, max=10, step=1, interval=400, description="Pace")
position_slider = widgets.IntSlider(value=0, min=0, max=10, description="Position:",
                                    style={"description_width": "120px"},
                                    layout=widgets.Layout(width="300px"))
widgets.jslink((play_button, "value"), (position_slider, "value"))
widgets.jslink((play_button, "max"), (position_slider, "max"))

prev_btn = widgets.Button(description="◄ Previous", layout=widgets.Layout(width="110px"))
next_btn = widgets.Button(description="Next ►",    layout=widgets.Layout(width="110px"))
reset_btn = widgets.Button(description="↺ Restart", layout=widgets.Layout(width="110px"))

# ---------------------------------------------------------------------------
# Renderer
# ---------------------------------------------------------------------------
output_area = widgets.Output()

def render(_change=None):
    with output_area:
        clear_output(wait=True)

        passage_text = PASSAGES[passage_dd.value]
        font_family = (
            '"OpenDyslexic", "Comic Sans MS", system-ui, sans-serif'
            if font_toggle.value == "OpenDyslexic"
            else 'system-ui, -apple-system, "Segoe UI", Roboto, sans-serif'
        )

        chunks = split_into_chunks(passage_text, chunk_size_slider.value)
        new_max = max(0, len(chunks) - 1)
        if position_slider.max != new_max:
            position_slider.max = new_max
            play_button.max = new_max
            if position_slider.value > new_max:
                position_slider.value = new_max

        # Pacer interval: ms per chunk = (chunk_size / WPM) * 60000
        interval_ms = int((chunk_size_slider.value / max(1, wpm_slider.value)) * 60000)
        play_button.interval = max(80, interval_ms)

        base_style = (
            f"font-family: {font_family}; "
            "font-size: 22px; line-height: 1.9; letter-spacing: 0.01em; "
            "padding: 28px 32px; max-width: 720px; "
            "background: #fbf9f3; color: #2c2a26; "
            "border-radius: 12px; border: 1px solid #ece6d3;"
        )

        if mode_toggle.value == "free":
            body = passage_text
            html = f'<div style="{base_style}">{body}</div>'
        else:
            current = position_slider.value
            parts = []
            for i, chunk in enumerate(chunks):
                if i == current:
                    parts.append(
                        '<mark style="background:#ffe97a; padding:2px 6px; '
                        'border-radius:6px; box-decoration-break: clone; '
                        '-webkit-box-decoration-break: clone;">'
                        f'{chunk}</mark>'
                    )
                elif i < current:
                    parts.append(f'<span style="color:#a8a39a;">{chunk}</span>')
                else:
                    parts.append(f'<span style="color:#cfc9bb;">{chunk}</span>')
            html = f'<div style="{base_style}">{" ".join(parts)}</div>'
            html += (
                '<p style="font-family: system-ui, sans-serif; color:#6b6660; '
                'margin: 10px 4px 0; font-size: 14px;">'
                f'Chunk {current + 1} of {len(chunks)} · '
                f'≈ {wpm_slider.value} WPM · '
                f'{chunk_size_slider.value} words at a time</p>'
            )

        display(HTML(html))

# Wire change observers
for w in (passage_dd, font_toggle, mode_toggle, wpm_slider, chunk_size_slider, position_slider):
    w.observe(render, names="value")

# Button handlers
def on_prev(_b):
    position_slider.value = max(0, position_slider.value - 1)
def on_next(_b):
    position_slider.value = min(position_slider.max, position_slider.value + 1)
def on_reset(_b):
    position_slider.value = 0

prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
reset_btn.on_click(on_reset)

# ---------------------------------------------------------------------------
# Layout
# ---------------------------------------------------------------------------
section_label = lambda t: widgets.HTML(
    f'<div style="font-family: system-ui; font-weight: 600; color:#444; '
    f'margin: 14px 0 4px;">{t}</div>'
)

controls = widgets.VBox([
    section_label("Choose your reading"),
    passage_dd,
    section_label("Customize the look"),
    font_toggle,
    section_label("Choose how to read"),
    mode_toggle,
    wpm_slider,
    chunk_size_slider,
    section_label("Pacer controls (guided mode)"),
    widgets.HBox([play_button, position_slider]),
    widgets.HBox([prev_btn, next_btn, reset_btn]),
])

display(controls, output_area)
render()

## Why these two features

**OpenDyslexic font.** OpenDyslexic was designed specifically to reduce letter rotation, mirroring, and crowding — the most-cited visual sources of friction for dyslexic readers. It is not a cure, and the empirical evidence for any single typeface is mixed, but giving learners the *choice* is the point. Most mainstream EdTech platforms in India force a single typeface chosen for brand consistency, not legibility. A toggle returns that decision to the reader.

**Reading pace control.** Mainstream platforms move at the page's pace, not the learner's. For ADHD learners, that often means a wall of text appearing all at once and triggering shutdown. For dyslexic learners, it means re-reading the same line three times because the visual baseline keeps slipping. The guided pacer chunks the passage into short, manageable units and advances at a speed the learner sets. There is no "fall behind." There is no streak. The interface is the intervention.

## Future roadmap (descoped for MP2)

The MP2a declaration named six features. After scoping feedback, four are deferred to a v2 / future-work track so this build could ship something complete in two weeks:

1. **Color contrast customization** — selectable background/foreground pairings (warm beige, dark mode, blue-tinted) for visual-stress users.
2. **Motion reduction** — a global toggle that removes any easing, fades, or animation, including the pacer highlight transition.
3. **Audio-synchronized reading** — text-to-speech narration with the current word highlighting in time with playback. This is the hardest interaction on the list and would have eaten the entire two weeks on its own.
4. **Self-paced progress milestones** — local-only celebration of individual reading gains (passages completed, time spent reading), with zero comparison or streak pressure.

Each of these is a meaningful design problem in its own right. Documenting them as roadmap, not features, is itself the scoping decision the assignment asked for.

## Credits & licensing

- **Reading passages:** Aesop's fables, public domain (Townsend / Jacobs English translations, lightly adapted).
- **OpenDyslexic font:** SIL Open Font License (free for any use). Source: https://opendyslexic.org
- **Built for:** HCDE 530 (Spring 2026), Mini Project 2.
- **Audience:** Neurodivergent learners ages 10–16, particularly in Indian classrooms; secondary audience educators supporting differentiated instruction.